<div align="center">

# 🚕 Yandex Go Analytics

## 2 · Batch Ingestion

*Read Parquet and Avro files straight from the Volume and persist them in Bronze — as they are.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![ingestion](https://img.shields.io/badge/ingestion-Auto%20Loader-FF3621)
![formats](https://img.shields.io/badge/formats-Parquet%20%2B%20Avro-00ADD4)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [1 · Setup](./1.%20Setup.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) |

**Progress** &nbsp; ▰▰▱▱▱▱▱▱  **2 / 8**

---

## 🎯 Task requirements

- [x] Read the **`.parquet`** and **`.avro`** files directly from the Volume
- [x] Write them **"as-is"** into the `bronze` schema as **Delta** tables: `bronze.taxi` and `bronze.users`

## 🧩 How it is solved

Both tables use the same service, `BaseIngestionService`:

| Step | What happens |
|:--|:--|
| 1 · **Extract** | Auto Loader (`cloudFiles`) reads new files from the Volume; the schema is inferred and stored next to the checkpoint |
| 2 · **Audit columns** | `_ingested_at` (timestamp) and `_source_file` (path) are added — the business columns stay untouched |
| 3 · **Load** | Append to a Delta table with `trigger(availableNow=True)`: process everything available, then stop |

| Entity | Source | Target |
|:--|:--|:--|
| 🚕 Trips | `…/landing/parquet` | `bronze.taxi` |
| 👤 Users | `…/landing/avro` | `bronze.users` |

> 💡 **Incremental by design** — the checkpoint remembers which files were processed, so a re-run ingests only
> *new* files and never duplicates data. Auto Loader also adds a `_rescued_data` column that captures values
> that do not match the inferred schema.

## 1️⃣ Connect to Spark

In [9]:
from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-batch-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 17:44:13.995 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-batch-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_prod | env = prod


## 2️⃣ Peek at the raw sources

Spark can read the files directly from the Volume path — no copy, no staging.

### 🚕 Parquet — trips

In [10]:
SRC_PARQUET = f"/Volumes/{CATALOG}/raw_files/landing/parquet"
SRC_AVRO = f"/Volumes/{CATALOG}/raw_files/landing/avro"

source_trips = spark.read.parquet(SRC_PARQUET)
print(f"Parquet source: {source_trips.count():,} rows | {len(source_trips.columns)} columns")
display(source_trips.toPandas())

Parquet source: 10,000 rows | 22 columns


,id,user_id,driver_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,dda1b2ca-f782-4294-ade5-bcc2bcf6646f,94a1875d-2db6-4edb-82de-ffccf86c2ca2,b4f741aa-e8c7-4c9e-92f1-3d67d8ed6793,2,2026-10-02 13:26:41,2026-10-02 13:41:41,3,28.33,1,Y,...,1,112.03,0.5,0.5,16.80,10.30,1.0,141.13,0.0,0.00
1,ce416144-c6f3-4685-8753-38846fb43f1d,1238d630-743b-45a2-9c0f-2fcfb3f6fe0d,6af29145-404f-48ed-9b5b-81d79469a394,2,2026-10-08 17:39:50,2026-10-08 18:15:50,3,0.00,5,N,...,3,18.64,0.0,0.5,2.80,0.00,1.0,25.44,2.5,0.00
2,760fa9c9-62cd-42ae-a3b6-2b843617df02,ded255d0-bf1e-4366-8b8e-63d4ce7607ad,fff569c2-ee2b-48e4-93d1-622569a81137,2,2026-10-05 18:03:29,2026-10-05 18:22:29,3,5.66,1,N,...,2,22.46,0.0,0.5,3.37,0.00,1.0,31.08,2.5,1.25
3,d9340848-b46c-45a2-9b75-9c5f036eb505,7cea2045-c268-483e-a32f-2e63b7fddd71,258d5a93-958e-4aa7-8dfe-519dc24c472e,2,2026-10-04 22:25:27,2026-10-04 22:59:27,3,2.18,1,N,...,1,10.92,0.5,0.5,1.64,0.00,1.0,17.06,2.5,0.00
4,fba2525e-cf22-474b-ba2f-c7f2385f6824,65c17795-b155-46bc-9f8d-ed9756abf2f1,b26e24e0-e412-4f89-a8b0-683224c6435c,1,2026-10-08 17:28:53,2026-10-08 18:05:53,0,23.52,1,N,...,3,92.26,0.5,0.5,13.84,7.91,1.0,118.51,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,ac4adfa0-46db-43e1-bac3-21c66716d861,1d34d08e-7a4c-45d4-9c99-e04cf0e98b3b,f3a3c571-7476-4899-b5a3-adb3254a9493,1,2026-09-30 21:15:18,2026-09-30 21:49:18,1,4.84,1,Y,...,4,20.72,0.5,0.5,3.11,9.62,1.0,37.95,2.5,0.00
9996,86236867-eadb-481c-8b55-48133d7dd5dd,269cd696-236c-4b87-94a0-bccb8a476a87,6576923c-81e1-4733-91dd-85fdb4164b76,2,2026-10-06 01:52:30,2026-10-06 02:20:30,3,17.36,2,Y,...,1,52.01,0.5,0.5,0.00,0.00,1.0,56.51,2.5,0.00
9997,34a9cbe2-74b9-4ba7-852a-58179a24d139,e0ccedc5-f05d-476e-9a84-a51aa9d3d7c7,fd35bbb7-fb13-4e8b-84dc-57de990e75ea,1,2026-10-01 07:31:37,2026-10-01 07:34:37,6,25.74,5,Y,...,1,95.81,0.0,0.5,14.37,0.00,1.0,115.43,2.5,1.25
9998,ea25b1c5-eb46-4a33-a298-383a8c922542,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,0c119c08-391c-4954-9dba-78d2943c10ee,2,2026-10-04 15:39:13,2026-10-04 16:03:13,3,8.92,2,N,...,3,33.05,0.0,0.5,4.96,6.50,1.0,48.51,2.5,0.00


### 👤 Avro — users

In [11]:
source_users = spark.read.format("avro").load(SRC_AVRO)
print(f"Avro source: {source_users.count():,} rows | {len(source_users.columns)} columns")
display(source_users.toPandas())

Avro source: 2,500 rows | 5 columns


,id,full_name,email,registration_date,is_plus_subscriber
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Pamela Johnson,martinphillip@example.com,2021-05-12,True
1,23b8c1e9-3924-46de-beb1-3b9046685257,Jeanne Edwards,emily13@example.net,2022-01-21,True
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,Wendy Valdez,michael96@example.net,2022-04-07,True
3,972a8469-1641-4f82-8b9d-2434e465e150,Travis White,mariahwhite@example.com,2021-02-03,True
4,17fc695a-07a0-4a6e-8822-e8f36c031199,Kaitlyn Foster,johnsonbenjamin@example.org,2020-11-07,True
...,...,...,...,...,...
2495,d56ce8ea-1959-4b5a-a7a8-f6366a35df59,Keith Clements,morgansteven@example.com,2026-04-02,True
2496,f065c817-687a-4850-9d9b-62317d45d8ef,Roberto Maldonado,ycardenas@example.org,2020-04-24,True
2497,b0b63bcf-0860-4833-879d-0cdaf396ea37,Elizabeth Robertson,brooke88@example.net,2022-12-17,True
2498,71d79665-7181-4dcf-b79e-fc6e5edb0d3c,Dr. Sarah Brown DDS,murraykirsten@example.com,2021-07-07,False


## 3️⃣ Ingest into Bronze

Two services, one pattern. Each call blocks until the available files are fully processed.

```text
 landing/avro      ──▶ UserBatchIngestionService      ──▶ bronze.users
 landing/parquet   ──▶ TaxiTripBatchIngestionService  ──▶ bronze.taxi
```

In [12]:
from src.core.etl.bronze import (
    TaxiTripBatchIngestionService,
    UserBatchIngestionService,
)

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

print("✅ Batch ingestion finished")

2026-10-09 17:44:20.698 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_prod.bronze.users
2026-10-09 17:44:20.700 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [avro] from: /Volumes/yandex_go_prod/raw_files/landing/avro
2026-10-09 17:44:20.702 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_prod.bronze.users
2026-10-09 17:44:28.567 | INFO     | src.core.etl.bronze.base:load:111 - Successfully loaded in yandex_go_prod.bronze.users
2026-10-09 17:44:28.570 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_prod.bronze.taxi
2026-10-09 17:44:28.572 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [parquet] from: /Volumes/yandex_go_prod/raw_files/landing/parquet
2026-10-09 17:44:28.574 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_prod.bronze.taxi
2026-10-09 17:44:36.485 | INFO     | src.core.etl.bronze.base:load:1

✅ Batch ingestion finished


---

## ✅ Checkpoint

- [x] Parquet and Avro files read directly from the Volume
- [x] `bronze.taxi` and `bronze.users` written as Delta tables
- [x] Row counts match the sources; re-running adds nothing

> **Next up ▶ [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb)** — stream drivers into Bronze, evolve the schema and travel through time.

<div align="center"><sub>Yandex Go Analytics · notebook 2 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>